# PennyLane QROM Test

This notebook tests `qrom_table_2d` from `integrations.pennylane.block_encoding.qrom`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pennylane as qml

# Add repo src/ to import path
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
repo_root = None
for c in candidates:
    if (c / 'src').exists():
        repo_root = c
        break
if repo_root is None:
    raise RuntimeError('Could not find repo root containing src/.')
sys.path.insert(0, str(repo_root / 'src'))

from integrations.pennylane.block_encoding.qrom import qrom_table_2d

In [2]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
], dtype=int)

i_wires = [0]          # 1 bit for N=2
j_wires = [1, 2]       # 2 bits for M=3
data_wires = [3, 4, 5] # 3 bits for max value=6

dev = qml.device('default.qubit', wires=6)

In [3]:
def bits(value: int, width: int):
    return np.array([int(b) for b in format(value, f'0{width}b')], dtype=int)

@qml.qnode(dev)
def load_value(i_val: int, j_val: int):
    qml.BasisState(bits(i_val, len(i_wires)), wires=i_wires)
    qml.BasisState(bits(j_val, len(j_wires)), wires=j_wires)
    # data register starts in |000>
    qrom_table_2d(A, i_wires=i_wires, j_wires=j_wires, data_wires=data_wires)
    return qml.probs(wires=data_wires)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        probs = load_value(i, j)
        loaded = int(np.argmax(probs))
        print(f'(i={i}, j={j}) -> loaded {loaded}, expected {A[i,j]}')

(i=0, j=0) -> loaded 1, expected 1
(i=0, j=1) -> loaded 2, expected 2
(i=0, j=2) -> loaded 3, expected 3
(i=1, j=0) -> loaded 4, expected 4
(i=1, j=1) -> loaded 5, expected 5
(i=1, j=2) -> loaded 6, expected 6


In [ ]:
# Strict check
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        loaded = int(np.argmax(load_value(i, j)))
        assert loaded == int(A[i, j])
print('All checks passed.')

All checks passed.


: 